# Walk-Forward Weighted Objective v1

Purpose: turn the recent gate-gap conclusion into a reusable candidate selector diagnostic. This notebook scores existing candidates against the active target: excess return >= 30pp and MaxDD <= 15%, with walk-forward windows carrying explicit weight.

This is diagnostic only. It does not change runtime auto-selection or bind a new production model.

## External Basis

- [Qlib GitHub](https://github.com/microsoft/qlib): keep selection inside a reproducible quant workflow.
- [Qlib arXiv 2009.11189](https://arxiv.org/abs/2009.11189): model selection must be part of the dataset-model-backtest-analysis loop.
- [Probability of Backtest Overfitting](https://www.davidhbailey.com/dhbpapers/backtest-overfitting.pdf): strict/OOS screens can overfit when many candidates are tried; walk-forward evidence must govern the next decision.
- [Deflated Sharpe Ratio](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551): multiple-testing corrections motivate ranking by robust gate gaps instead of headline return.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.walkforward_selection import (
    render_walkforward_selection_review,
    score_walkforward_candidates,
    summarize_walkforward_selection,
)

ROOT = Path.cwd()
CARD_DIR = ROOT / 'runtime/model_selection/card/walkforward_weighted_objective_v1'
SOURCE_DIR = ROOT / 'runtime/model_selection/card/selection_overfit_audit_v1'
WF_PATH = SOURCE_DIR / 'walkforward_daily_screen.csv'
STRICT_PATH = SOURCE_DIR / 'strict_daily_target_screen.csv'
SUMMARY_PATH = SOURCE_DIR / 'summary_selection_overfit_table.csv'

TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0

external_references = [
    {'label': 'Qlib', 'url': 'https://github.com/microsoft/qlib', 'note': 'reproducible quant workflow'},
    {'label': 'Qlib arXiv', 'url': 'https://arxiv.org/abs/2009.11189', 'note': 'AI-oriented quantitative investment workflow'},
    {'label': 'Probability of Backtest Overfitting', 'url': 'https://www.davidhbailey.com/dhbpapers/backtest-overfitting.pdf', 'note': 'multiple testing and walk-forward guard'},
    {'label': 'Deflated Sharpe Ratio', 'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551', 'note': 'selection-bias adjusted performance evidence'},
]


In [2]:
wf = pd.read_csv(WF_PATH)
strict = pd.read_csv(STRICT_PATH)
summary_source = pd.read_csv(SUMMARY_PATH)

metrics_by_candidate: dict[str, dict[str, dict[str, float]]] = {}
source_rows = []
for candidate, sub in wf.groupby('candidate'):
    metrics_by_candidate[candidate] = {}
    for idx, row in sub.reset_index(drop=True).iterrows():
        window = str(row.get('wf') or row.get('card') or f'window_{idx}')
        if window in metrics_by_candidate[candidate]:
            window = f"{window}_{idx}"
        metrics_by_candidate[candidate][window] = {
            'return': float(row['return']),
            'buy_and_hold_return': float(row['buyhold_return']),
            'relative_return': float(row['relative_return']),
            'sharpe': float(row['sharpe']),
            'max_dd': float(row['max_dd']),
            'avg_pos': float(row['avg_pos']),
            'trades': float(row['trades']),
        }
        source_rows.append({'candidate': candidate, 'window': window, 'path': row['path']})

source_coverage = pd.DataFrame(source_rows)
source_coverage.to_csv(CARD_DIR / 'source_coverage.csv', index=False)
display(source_coverage.groupby('candidate').agg(windows=('window', 'count')).reset_index())


,candidate,windows
0,balanced_rule_residual_imitation_v4,3
1,balanced_rule_residual_mode_v1,3
2,recovery_reentry_2021_trendbreak_cap_v1,3
3,recovery_reentry_anchor_confirm_v1,3
4,recovery_reentry_conditional_vol_target_v1,3
5,recovery_reentry_cppi_floor_v1,3
6,recovery_reentry_equity_drawdown_guard_v1,3
7,recovery_reentry_floor_v1,3
8,recovery_reentry_regime_drawdown_guard_v1,3
9,risk_budget_residual_drawdown_guard_walkforwar...,12


In [3]:
scored = score_walkforward_candidates(metrics_by_candidate)
selection_summary = summarize_walkforward_selection(scored)

scored.to_csv(CARD_DIR / 'walkforward_weighted_scores.csv', index=False)
(CARD_DIR / 'selection_summary.json').write_text(json.dumps(selection_summary, ensure_ascii=False, indent=2), encoding='utf-8')

display(scored)
display(selection_summary)


,candidate,window_count,gate_pass_count,full_gate,weighted_objective_loss,mean_relative_return,min_relative_return,max_max_dd,mean_avg_pos,dominant_failure_type,worst_window,next_mechanism
0,recovery_reentry_conditional_vol_target_v1,3,0,False,237.495543,63.221576,55.796392,26.139320,0.282693,drawdown_control,wf_2020,drawdown_state_guard_or_sequence_risk_feature
1,recovery_reentry_cppi_floor_v1,3,0,False,276.141700,62.571551,55.796392,30.183184,0.303902,drawdown_control,wf_2023_2024,drawdown_state_guard_or_sequence_risk_feature
2,recovery_reentry_regime_drawdown_guard_v1,3,0,False,378.452732,61.269911,55.796392,30.183184,0.314326,drawdown_control,wf_2023_2024,drawdown_state_guard_or_sequence_risk_feature
3,recovery_reentry_2021_trendbreak_cap_v1,3,0,False,384.594038,44.753732,25.557493,28.204933,0.279729,drawdown_control,wf_2023_2024,drawdown_state_guard_or_sequence_risk_feature
4,soxl_soxx_efficiency_chop_guard_v1,3,0,False,385.932397,42.024165,25.816839,30.512835,0.285571,drawdown_control,wf_2023_2024,drawdown_state_guard_or_sequence_risk_feature
5,recovery_reentry_floor_v1,3,0,False,401.915992,57.416421,49.902600,30.183184,0.327356,drawdown_control,wf_2023_2024,drawdown_state_guard_or_sequence_risk_feature
6,recovery_reentry_anchor_confirm_v1,3,0,False,408.862913,60.084194,55.796392,30.183184,0.314751,drawdown_control,wf_2023_2024,drawdown_state_guard_or_sequence_risk_feature
7,v19reg_v22_budget_usage_bull_guard_v1,3,0,False,408.862913,54.877823,40.177280,30.183184,0.306417,drawdown_control,wf_2023_2024,drawdown_state_guard_or_sequence_risk_feature
8,recovery_reentry_equity_drawdown_guard_v1,3,0,False,420.482443,31.028280,-15.413925,28.254676,0.260753,drawdown_control,wf_2020,drawdown_state_guard_or_sequence_risk_feature
9,v19reglight_v22_budget_usage_bull_guard_ablati...,3,0,False,452.676655,45.747521,33.693150,36.914320,0.290708,drawdown_control,wf_2023_2024,drawdown_state_guard_or_sequence_risk_feature


{'decision': 'no_full_gate_candidate',
 'selected_candidate': 'recovery_reentry_conditional_vol_target_v1',
 'selected_weighted_objective_loss': 237.49554277112543,
 'selected_dominant_failure_type': 'drawdown_control',
 'next_mechanism': 'walk_forward_weighted_multi_objective_training'}

In [4]:
strict_pass = strict[strict.get('target_pass', False) == True].copy()
strict_pass_candidates = sorted(strict_pass['candidate'].astype(str).unique()) if not strict_pass.empty else []
walkforward_candidates = set(scored['candidate'].astype(str))
strict_candidate_aliases = {}
for candidate in strict_pass_candidates:
    aliases = [candidate]
    if candidate.endswith('_v1'):
        aliases.append(candidate.replace('_v1', '_walkforward_v1'))
    matched = [alias for alias in aliases if alias in walkforward_candidates]
    strict_candidate_aliases[candidate] = matched
matched_walkforward_candidates = sorted({alias for aliases in strict_candidate_aliases.values() for alias in aliases})
unmatched_strict_candidates = sorted([candidate for candidate, aliases in strict_candidate_aliases.items() if not aliases])
strict_pass_score = scored[scored['candidate'].isin(matched_walkforward_candidates)].copy()
strict_pass_score.to_csv(CARD_DIR / 'strict_pass_walkforward_readout.csv', index=False)

overfit_readout = {
    'strict_pass_candidates': strict_pass_candidates,
    'strict_pass_count': int(len(strict_pass_candidates)),
    'matched_walkforward_candidates': matched_walkforward_candidates,
    'unmatched_strict_candidates': unmatched_strict_candidates,
    'strict_pass_full_walkforward_count': int(strict_pass_score['full_gate'].sum()) if not strict_pass_score.empty else 0,
    'strict_pass_best_loss': float(strict_pass_score['weighted_objective_loss'].min()) if not strict_pass_score.empty else None,
}
(CARD_DIR / 'strict_vs_walkforward_overfit_readout.json').write_text(
    json.dumps(overfit_readout, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

display(pd.DataFrame([overfit_readout]))
display(strict_pass_score)


,strict_pass_candidates,strict_pass_count,matched_walkforward_candidates,unmatched_strict_candidates,strict_pass_full_walkforward_count,strict_pass_best_loss
0,[risk_budget_residual_drawdown_guard_v1],1,[risk_budget_residual_drawdown_guard_walkforwa...,[],0,855.033985


,candidate,window_count,gate_pass_count,full_gate,weighted_objective_loss,mean_relative_return,min_relative_return,max_max_dd,mean_avg_pos,dominant_failure_type,worst_window,next_mechanism
13,risk_budget_residual_drawdown_guard_walkforwar...,12,3,False,855.033985,27.073475,-54.194588,28.569218,0.158467,joint_tradeoff,wf_2020,walk_forward_weighted_multi_objective_training


In [5]:
review = render_walkforward_selection_review(scored, selection_summary)
(CARD_DIR / 'walkforward_weighted_objective_review.md').write_text(review, encoding='utf-8')

selected = selection_summary['selected_candidate']
selected_row = scored[scored['candidate'] == selected].iloc[0].to_dict() if selected else {}
strict_count = overfit_readout['strict_pass_count']
strict_wf_count = overfit_readout['strict_pass_full_walkforward_count']

conclusion_md = f'''## Runtime Conclusion

Decision: `{selection_summary['decision']}`. Do not promote a candidate from this screen.

Key evidence:

- Selected diagnostic candidate by lowest walk-forward weighted loss: `{selected}`.
- Selected candidate still fails full target: `full_gate={selected_row.get('full_gate')}`, `max_max_dd={selected_row.get('max_max_dd', 0):.2f}%`, `min_relative_return={selected_row.get('min_relative_return', 0):.2f}pp`, `loss={selected_row.get('weighted_objective_loss', 0):.2f}`.
- Strict-screen pass count: `{strict_count}`; strict candidates that also pass all walk-forward windows: `{strict_wf_count}`.
- This is direct overfit evidence: strict OOS can pass while walk-forward windows still fail the <=15% drawdown target.

Next valid mechanism:

- Keep `{selected}` as the closest diagnostic reference, but do not bind it to runtime.
- Build the next branch as a walk-forward-weighted multi-objective training/selection objective that optimizes relative return and drawdown jointly across `wf_2020`, `wf_2022`, and `wf_2023_2024`.
- Because top candidates are mostly drawdown-control failures, combine state-conditioned drawdown guard with rebound participation pressure instead of a single global drawdown cap or another reward penalty sweep.
'''
(CARD_DIR / 'runtime_conclusion.md').write_text(conclusion_md, encoding='utf-8')

selection = {
    'experiment': 'walkforward_weighted_objective_v1',
    'decision': selection_summary['decision'],
    'selected_diagnostic_candidate': selected,
    'selected_weighted_objective_loss': selection_summary.get('selected_weighted_objective_loss'),
    'selected_dominant_failure_type': selection_summary.get('selected_dominant_failure_type'),
    'next_mechanism': selection_summary.get('next_mechanism'),
    'target': {
        'relative_return_min_pp': TARGET_RELATIVE_RETURN,
        'max_dd_max_pct': TARGET_MAX_DD,
    },
    'strict_vs_walkforward': overfit_readout,
    'promotion': 'blocked_no_full_gate_candidate',
    'artifacts': {
        'notebook': str((CARD_DIR / 'walkforward_weighted_objective_v1.ipynb').relative_to(ROOT)),
        'source_coverage': str((CARD_DIR / 'source_coverage.csv').relative_to(ROOT)),
        'scores': str((CARD_DIR / 'walkforward_weighted_scores.csv').relative_to(ROOT)),
        'selection_summary': str((CARD_DIR / 'selection_summary.json').relative_to(ROOT)),
        'strict_pass_readout': str((CARD_DIR / 'strict_pass_walkforward_readout.csv').relative_to(ROOT)),
        'strict_vs_walkforward': str((CARD_DIR / 'strict_vs_walkforward_overfit_readout.json').relative_to(ROOT)),
        'review': str((CARD_DIR / 'walkforward_weighted_objective_review.md').relative_to(ROOT)),
        'runtime_conclusion': str((CARD_DIR / 'runtime_conclusion.md').relative_to(ROOT)),
    },
    'external_references': external_references,
}
(CARD_DIR / 'selection.json').write_text(json.dumps(selection, ensure_ascii=False, indent=2), encoding='utf-8')

display(Markdown(conclusion_md))


## Runtime Conclusion

Decision: `no_full_gate_candidate`. Do not promote a candidate from this screen.

Key evidence:

- Selected diagnostic candidate by lowest walk-forward weighted loss: `recovery_reentry_conditional_vol_target_v1`.
- Selected candidate still fails full target: `full_gate=False`, `max_max_dd=26.14%`, `min_relative_return=55.80pp`, `loss=237.50`.
- Strict-screen pass count: `1`; strict candidates that also pass all walk-forward windows: `0`.
- This is direct overfit evidence: strict OOS can pass while walk-forward windows still fail the <=15% drawdown target.

Next valid mechanism:

- Keep `recovery_reentry_conditional_vol_target_v1` as the closest diagnostic reference, but do not bind it to runtime.
- Build the next branch as a walk-forward-weighted multi-objective training/selection objective that optimizes relative return and drawdown jointly across `wf_2020`, `wf_2022`, and `wf_2023_2024`.
- Because top candidates are mostly drawdown-control failures, combine state-conditioned drawdown guard with rebound participation pressure instead of a single global drawdown cap or another reward penalty sweep.
